# Exercise — GW190814: matched filtering in three detectors

**Goal:** Apply the methods learned from the GW150914 to **GW190814** using the local **H1, L1, and V1** strain files.

You will estimate a PSD for each detector, compute matched-filter SNRs, calculate the Allen $\chi^2$ statistic, and construct the reweighted SNR. You may need to use code from your earlier notebook.

**Event time:** GPS $1249852257.0$ (2019-08-14 21:10:39 UTC). The downloaded files cover 256 seconds at 4096 Hz, centred approximately on this time.

**Template starting point:** $m_1\simeq24.4\,M_\odot$, $m_2\simeq2.73\,M_\odot$ (approximate **detector-frame** masses). 

Use zero spins initially and the waveform approximant demonstrated earlier. 

The observed signal has appreciable higher-order-mode contributions, so this simple template is an exercise, **not** a fully optimized parameter-estimation waveform.

**Suggested analysis cutoff:** $f_{\rm{low}}=20$ Hz. Work in a short interval around the trigger when identifying peaks; ignore filtering-corrupted edges.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from pycbc.frame import read_frame
from pycbc.filter import highpass, resample_to_delta_t, matched_filter
from pycbc.psd import welch, interpolate, inverse_spectrum_truncation
from pycbc.waveform import get_td_waveform
from pycbc.vetoes import power_chisq
from pycbc.events.ranking import newsnr

plt.rcParams.update({'figure.figsize': (11, 4), 'font.size': 11})

/home/abhishek.sharma/miniconda3/envs/gw-env/lib/python3.11/site-packages/pycbc/types/array.py:36: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal as _lal
PyCBC.libutils: pkg-config call failed, setting NO_PKGCONFIG=1


In [3]:
GPS = 1249852257.0
SAMPLING_FREQ = 2048
DURATION = 256
IFOS = ['H1', 'L1', 'V1']

# Works when launched from either repository root or notebooks/.
DATA_DIR = Path('data') if Path('data').is_dir() else Path('../data')

def load_detector(ifo):
    """Read locally prepared GW190814 GWF and condition the strain."""
    filename = f'GW190814_{ifo}_4096Hz_{DURATION}s.gwf'
    path = DATA_DIR / filename
    channel = f'{ifo}:GWOSC-4KHZ_R1_STRAIN'

    if not path.exists():
        raise FileNotFoundError(f'{path} is missing. Run python download_data.py first.')

    raw = read_frame(str(path), channel)
    strain = resample_to_delta_t(highpass(raw, 15.0), 1.0 / SAMPLING_FREQ)
    return strain.crop(2, 2)


strain = {ifo: load_detector(ifo) for ifo in IFOS}
for ifo in IFOS:
    data = strain[ifo]
    print(f'{ifo}: start={data.start_time}, duration={data.duration}, fs={data.sample_rate}')

H1: start=1249852131, duration=252.0, fs=2048.0
L1: start=1249852131, duration=252.0, fs=2048.0
V1: start=1249852131, duration=252.0, fs=2048.0


## Task 1 — Compute PSDs for H1, L1, and V1

1. Use Welch's method with **4-second segments and 50% overlap** to estimate the noise PSD separately for each detector.
2. Interpolate each PSD to the frequency resolution of its corresponding strain data.
3. Apply inverse-spectrum truncation with a **4-second** impulse-response length and $f_{\rm low}=20$ Hz.
4. Plot the **amplitude spectral densities** ($\sqrt{S_n(f)}$) for all three detectors on one log–log plot.
5. Which detector has the best sensitivity?

**Check:** Why do the three detectors have different ASD curves? Why must each PSD match the frequency resolution of its detector's strain segment?

## Task 2 — Matched-filter SNR

1. Generate a waveform template using initial detector-frame component masses **$m_1=24.4\,M_\odot$ and $m_2=2.73\,M_\odot$**, zero spins, and $f_{\rm low}=20$ Hz.
2. Resize and cyclically shift the template as demonstrated in the GW150914 notebook to match the strain segment and the matched-filter time convention.
3. Matched-filter **each detector separately**, using its own PSD.
4. Plot $|\rho(t)|$ for H1, L1, and V1 near the expected event time. Record the peak SNR and GPS time for each detector.

**Check:** Do the detector peaks occur at exactly the same time? Why might their SNRs differ? Avoid circular-convolution and PSD-filtering edge regions when looking for peaks.

**Reminder:** A fixed template provides an illustrative result, not necessarily the published search SNR.

## Task 3 — Allen $\chi^2$ signal-consistency test

1. Use PyCBC's `power_chisq` with the **same template and detector-specific PSD** used above.
2. Choose a suitable number of frequency bins, for example **$p=16$**, and calculate the reduced statistic

$$\chi_r^2=\frac{\chi^2}{2p-2}.$$

3. Evaluate $\chi_r^2$ at the matched-filter peak found in **each detector**.
4. Plot $\chi_r^2(t)$ near the event.

**Check:** Is $\chi_r^2$ close to 1 in every detector? What could cause departures (noise, imperfect template, higher modes, etc.)?

## Task 4 — Reweighted SNR

1. Use `newsnr` to compute the reweighted SNR from $|\rho|$ and $\chi_r^2$.
2. Plot the ordinary SNR and reweighted SNR in a window near GW190814 for all three detectors.
3. Compare the peak values and summarize the results in a small table.

**Check:** When does $\chi^2$ penalize the SNR? Why can a genuine signal have $\chi_r^2$ noticeably different from 1 for a simplified template?

## Discussion questions

1. Which detector provides the clearest evidence for GW190814? Is this determined by sensitivity alone?
2. What differences do you observe between this asymmetric-mass event and GW150914?
3. Why would a real search use a **bank of templates**, rather than just one template?

**References:** [GWOSC GW190814 event page](https://gwosc.org/eventapi/html/O3_Discovery_Papers/GW190814/v1/)